# CheXpert Scientific Protocol: Kaggle Training Notebook

This notebook trains a single architecture and seed (`convnext_small` or `densenet121`, seed 42..46) under **Protocol v0.1**.

### Strict Protocol Rules
- **No Data Leakage**: Partitions only CheXpert `train.csv` at the patient level into Train (80%), Calibration (10%), and Internal-Validation (10%).
- **No Official Validation**: The official CheXpert `valid.csv` is sealed as locked test and must NEVER be accessed during training or calibration.
- **Single Run**: Runs exactly one architecture and one random seed per session.


In [ ]:
# ==============================================================================
# EXPERIMENT CONFIGURATION
# ==============================================================================
REPO_REF = "main"  # Commit SHA or release tag (e.g. "main")
DATA_ROOT = "/kaggle/input/chexpert-v10-small"
WORK_DIR = "/kaggle/working/chex"
ARCH = "convnext_small"  # "convnext_small" or "densenet121"
SEED = 42                # 42, 43, 44, 45, 46
RUN_MODE = "smoke"       # "smoke" (1 epoch quick check) or "full" (20 epochs protocol)
RESUME_CHECKPOINT = None # Optional: Path to checkpoint .pt to resume training

assert ARCH in ["convnext_small", "densenet121"], f"Invalid ARCH: {ARCH}"
assert SEED in [42, 43, 44, 45, 46], f"Invalid SEED: {SEED}"
assert RUN_MODE in ["smoke", "full"], f"Invalid RUN_MODE: {RUN_MODE}"
print(f"[CONFIG] Architecture: {ARCH} | Seed: {SEED} | Mode: {RUN_MODE}")


In [ ]:
# ==============================================================================
# CELL 1: Environment & Hardware Diagnostic
# ==============================================================================
import sys
import torch
import torchvision

print(f"Python Version   : {sys.version.split()[0]}")
print(f"PyTorch Version  : {torch.__version__}")
print(f"Torchvision      : {torchvision.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name  : {torch.cuda.get_device_name(0)}")
    print(f"cuDNN Version    : {torch.backends.cudnn.version()}")
    print(f"VRAM Total (GB)  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}")


In [ ]:
# ==============================================================================
# CELL 2: Repository Setup & Checkout
# ==============================================================================
import os
import subprocess
import shutil
from pathlib import Path

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

# Clone repository if not already cloned
if not (Path(WORK_DIR) / ".git").exists():
    print(f"Cloning repository into {WORK_DIR}...")
    subprocess.check_call(["git", "clone", "https://github.com/qdat2644/chex.git", "."])
    subprocess.check_call(["git", "checkout", REPO_REF])
else:
    print(f"Repository exists in {WORK_DIR}.")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Repository ready at Git commit: {commit}")


In [ ]:
# ==============================================================================
# CELL 3: Dependency Installation & Compilation Check
# ==============================================================================
import subprocess
import sys

print("Installing dependencies from requirements.txt...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "--quiet"])

# Verify core project packages directly
import torch, torchvision, pydicom, sklearn, scipy, yaml, pandas, numpy, PIL, fastapi, uvicorn
print(f"[OK] Core packages verified: PyTorch {torch.__version__}, SciPy {scipy.__version__}")

subprocess.check_call([sys.executable, "-m", "compileall", "-q", "app", "scripts", "tests"])
print("[OK] Environment compiled and dependencies verified.")


In [ ]:
# ==============================================================================
# CELL 4: Data Root Verification
# ==============================================================================
from pathlib import Path

data_root_path = Path(DATA_ROOT)
if not data_root_path.exists():
    # Fallback to local search in /kaggle/input
    candidates = list(Path("/kaggle/input").glob("**/train.csv"))
    if candidates:
        data_root_path = candidates[0].parent
        print(f"Found CheXpert data at: {data_root_path}")
    else:
        raise FileNotFoundError(f"Cannot find CheXpert train.csv in {DATA_ROOT} or /kaggle/input!")

train_csv = data_root_path / "train.csv" if (data_root_path / "train.csv").exists() else data_root_path / "CheXpert-v1.0-small" / "train.csv"
assert train_csv.is_file(), f"Missing train.csv at '{train_csv}'"
print(f"[OK] Training dataset located: {train_csv}")


In [ ]:
# ==============================================================================
# CELL 5: Patient-Level Partitioning Manifest Generation
# ==============================================================================
import subprocess
import sys
from pathlib import Path

manifest_dir = Path(WORK_DIR) / "outputs" / "splits" / "protocol_v0_1"
manifest_dir.mkdir(parents=True, exist_ok=True)
manifest_path = manifest_dir / "manifest.json"

if not manifest_path.is_file():
    print("Generating patient-level split manifest (80% train, 10% val, 10% calib)...")
    cmd = [
        sys.executable, "scripts/make_splits.py",
        "--data-root", str(data_root_path.parent if (data_root_path / 'CheXpert-v1.0-small').exists() else data_root_path),
        "--train-csv", str(train_csv),
        "--output-dir", str(manifest_dir),
        "--seed", "42",
        "--protocol-version", "0.1",
    ]
    subprocess.check_call(cmd)
else:
    print(f"Manifest already exists at: {manifest_path}")


In [ ]:
# ==============================================================================
# CELL 6: Split Anti-Leakage & Integrity Verification
# ==============================================================================
import json
import pandas as pd
from pathlib import Path

manifest_data = json.loads(manifest_path.read_text(encoding="utf-8"))
split_df = pd.read_csv(manifest_dir / manifest_data["splits_csv"])

train_pids = set(split_df[split_df["split"] == "train"]["patient_id"])
val_pids = set(split_df[split_df["split"] == "internal_validation"]["patient_id"])
cal_pids = set(split_df[split_df["split"] == "calibration"]["patient_id"])

assert len(train_pids & val_pids) == 0, "LEAKAGE DETECTED: Train & Val patient overlap!"
assert len(train_pids & cal_pids) == 0, "LEAKAGE DETECTED: Train & Calib patient overlap!"
assert len(val_pids & cal_pids) == 0, "LEAKAGE DETECTED: Val & Calib patient overlap!"

print(f"[PASSED] Zero patient leakage verified: {len(train_pids)} train, {len(val_pids)} val, {len(cal_pids)} calib patients.")
print(f"Manifest SHA-256: {manifest_data.get('manifest_sha256')}")


In [ ]:
# ==============================================================================
# CELL 7: Training (Single Architecture and Random Seed)
# ==============================================================================
import subprocess
import sys
from pathlib import Path

out_run_dir = Path(WORK_DIR) / "outputs" / "runs" / ARCH / f"seed_{SEED}"
config_file = Path(WORK_DIR) / "configs" / "protocol_v0_1.yaml"

train_cmd = [
    sys.executable, "scripts/train.py",
    "--manifest", str(manifest_path),
    "--config", str(config_file),
    "--arch", ARCH,
    "--seed", str(SEED),
    "--output-dir", str(out_run_dir),
]

if RUN_MODE == "smoke":
    print("Running SMOKE TEST (1 epoch, limited samples, NON_FINAL)...")
    train_cmd.extend(["--epochs", "1", "--limit", "64"])
else:
    print(f"Running FULL PROTOCOL TRAINING (20 epochs, Arch={ARCH}, Seed={SEED})...")

subprocess.check_call(train_cmd)

assert (out_run_dir / "best.pt").is_file(), f"Missing best.pt at {out_run_dir}"
assert (out_run_dir / "last.pt").is_file(), f"Missing last.pt at {out_run_dir}"
assert (out_run_dir / "run_metadata.json").is_file(), f"Missing run_metadata.json at {out_run_dir}"
print(f"[OK] Training completed -> {out_run_dir}")


In [ ]:
# ==============================================================================
# CELL 8: Resume Verification (Optional)
# ==============================================================================
if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).is_file():
    print(f"Verifying resume capabilities from: {RESUME_CHECKPOINT}")
    resume_cmd = [
        sys.executable, "scripts/train.py",
        "--manifest", str(manifest_path),
        "--config", str(config_file),
        "--arch", ARCH,
        "--seed", str(SEED),
        "--output-dir", str(out_run_dir),
        "--resume", str(RESUME_CHECKPOINT),
    ]
    subprocess.check_call(resume_cmd)
    print("[OK] Resume training verification passed.")
else:
    print("No RESUME_CHECKPOINT specified; skipping resume check.")


In [ ]:
# ==============================================================================
# CELL 9: Threshold Calibration on Calibration Partition Only
# ==============================================================================
import subprocess
import sys
from pathlib import Path

calib_dir = Path(WORK_DIR) / "outputs" / "calibration"
calib_dir.mkdir(parents=True, exist_ok=True)
calib_out = calib_dir / f"{ARCH}_seed{SEED}.json"
best_ckpt = out_run_dir / "best.pt"

print(f"Optimizing F1 thresholds on Calibration Split for {ARCH} seed {SEED}...")
calib_cmd = [
    sys.executable, "scripts/calibrate.py",
    "--checkpoint", str(best_ckpt),
    "--split-manifest", str(manifest_path),
    "--output", str(calib_out),
    "--seed", str(SEED),
]
if RUN_MODE == "smoke":
    calib_cmd.extend(["--limit", "64"])

subprocess.check_call(calib_cmd)
assert calib_out.is_file(), f"Missing calibration artifact at {calib_out}"
print(f"[OK] Calibration artifact produced -> {calib_out}")


In [ ]:
# ==============================================================================
# CELL 10: Artifact Packaging & Integrity Hash Ledger
# ==============================================================================
import hashlib
import json
import zipfile
from pathlib import Path

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

pkg_dir = Path("/kaggle/working/kaggle_artifacts")
pkg_dir.mkdir(parents=True, exist_ok=True)
zip_path = pkg_dir / f"{ARCH}_seed{SEED}.zip"

files_to_pack = [
    out_run_dir / "best.pt",
    out_run_dir / "last.pt",
    out_run_dir / "history.csv",
    out_run_dir / "run_metadata.json",
    calib_out,
]

checksums = {}
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_pack:
        if f.is_file():
            zf.write(f, arcname=f.name)
            checksums[f.name] = sha256_file(f)

manifest_file = pkg_dir / f"{ARCH}_seed{SEED}_checksums.json"
manifest_file.write_text(json.dumps({"architecture": ARCH, "seed": SEED, "run_mode": RUN_MODE, "files": checksums}, indent=2), encoding="utf-8")

print(f"\n[SUCCESS] Kaggle artifact packaged: {zip_path}")
print(json.dumps(checksums, indent=2))
